In [33]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** XX  
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line):

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

In [34]:
tickers = pd.read_csv('Tickers_Example.csv', header=None, names = ['Ticker'])

#created to filter all tickers based on assignment requirements. Returns a list of tickers that meet the requirements.
def filter_ticks (tickers, min_volume = 5000, min_days_per_month = 18):
    start_date = '2025-10-01'
    end_date = '2026-10-01'
    filtered = []

    for i in tickers['Ticker'].drop_duplicates(): #passes ticker column of the csv, dropping duplicates to avoid them appearing twice
        try:
            ticker = yf.Ticker(i)
            
            history = ticker.history(start=start_date, end=end_date)
            
            if (history.empty): #ticker~ is invalid or has no data for the specified date range
                continue

            vol = history['Volume']
            month = history.index.strftime('%Y-%m')  #masks to group by month
            avg_vol = vol[vol.groupby(month).transform('count') >= min_days_per_month] #filters data by making sure the number of days is greater than 18

            if ((avg_vol.mean() >= min_volume) or avg_vol.empty): #verifies stock has a daily volume of at least 5000 shares over the past year
                info = ticker.info #saves runtime by running this after the initial check
                market = info.get('market')
                if (market == 'us_market' or market == 'ca_market') and (info.get('currency') == 'USD' or info.get('currency') == 'CAD') and (info.get('quoteType') == 'EQUITY'): #verifies stock is listed on a US or Canadian exchange and is traded in USD or CAD
                    filtered.append(i)
        except:
            continue #in case ticker is invalid, ignore the current ticker

    return filtered
                    

filtered_tickers = filter_ticks(tickers)


def get_usd_cad_rate(): #converts price from USD to CAD
    start_date = '2026-11-20'
    end_date = '2026-11-21'
    return yf.Ticker('USDCAD=X').history(start=start_date, end=end_date)['Close'].iloc[0]

def get_close_prices(tickers, start_date = '2026-11-20', end_date = '2026-11-21'): #change start date to 2026-11-27 and end to 2026-11-28
    close_prices = pd.DataFrame()
    rate = get_usd_cad_rate()
    for ticker in filtered_tickers:
        try:
            data = yf.download(ticker, start=start_date, end=end_date, progress=False) #get data for all valid tickers
            if yf.Ticker(ticker).info.get('currency') == 'USD': #if price is listed in USD
                close_prices[ticker] = rate * data['Close']
            else:
                close_prices[ticker] = data['Close']
        except:
            continue

    return close_prices

all_close_prices = get_close_prices(filtered_tickers)
print(all_close_prices)

$AGN: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$CELG: possibly delisted; no timezone found
$MON: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found


                  AAPL        ABBV         ABT        ACN        AIG  \
Date                                                                   
2026-11-20  491.350937  410.510991  147.738051  317.23519  114.28901   

                  AMZN         AXP          BA        BAC  BB.TO  ...  \
Date                                                              ...   
2026-11-20  354.026281  432.743417  223.600823  67.471826   11.1  ...   

                 PYPL        QCOM     SHOP.TO   T.TO       TD.TO         TXN  \
Date                                                                           
2026-11-20  76.436514  215.066445  177.089996  15.86  162.050003  378.309816   

                   UNH         UNP        UPS        USB  
Date                                                      
2026-11-20  487.463861  412.074428  152.35666  80.395317  

[1 rows x 35 columns]


## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

In [35]:
budget_a = 1_000_000
number_of_stocks_a = 10
start_date = '2024-11-27'
end_date = '2026-11-27'
max_stock_weight = 0.15
max_sector_weight = 0.40
max_fee = 2.15
fee_per_share = 0.001


exchange_rate = get_usd_cad_rate()

sectors = {}
currencies = {}
buy_price_cad = {}

for ticker in filtered_tickers:
    try:
        info = yf.Ticker(ticker).info
    except Exception:
        continue
    if ticker in all_close_prices.columns and not pd.isna(all_close_prices[ticker].iloc[-1]):
        sectors[ticker] = info.get('sector', 'Unknown')
        currencies[ticker] = info.get('currency')
        buy_price_cad[ticker] = all_close_prices[ticker].iloc[-1]

monthly_closes = {}

for ticker in list(buy_price_cad) + ['^GSPC', 'USDCAD=X']:
    close = yf.Ticker(ticker).history(start=start_date, end=end_date)['Close']
    close.index = close.index.tz_localize(None)
    monthly_closes[ticker] = close.resample('ME').last()

monthly_prices = pd.DataFrame(monthly_closes)

for ticker in monthly_prices.columns:
    if currencies.get(ticker) == 'USD' or ticker == '^GSPC':
        monthly_prices[ticker] = monthly_prices[ticker] * monthly_prices['USDCAD=X']

monthly_returns = monthly_prices.pct_change().iloc[1:]
market_returns = monthly_returns['^GSPC']


rows = []

for ticker in buy_price_cad:
    expected_return = monthly_returns[ticker].mean()
    beta = monthly_returns[ticker].cov(market_returns) / market_returns.var()
    if not pd.isna(expected_return) and not pd.isna(beta):
        rows.append({'Ticker': ticker, 'Sector': sectors[ticker],
                     'Expected_Return': expected_return, 'Beta': beta})

stats_a = pd.DataFrame(rows).set_index('Ticker')
stats_a['Rank_ER'] = stats_a['Expected_Return'].rank(ascending=False)
stats_a['Rank_Beta'] = stats_a['Beta'].rank(ascending=False)
stats_a['Score'] = stats_a['Rank_ER'] + stats_a['Rank_Beta']
stats_a = stats_a.sort_values(['Score', 'Expected_Return'], ascending=[True, False])


max_per_sector = int(max_sector_weight * number_of_stocks_a)
picked_a = []
sector_count = {}

for ticker in stats_a.index:
    if len(picked_a) == number_of_stocks_a:
        break
    sector = stats_a.loc[ticker, 'Sector']
    if stats_a.loc[ticker, 'Expected_Return'] > 0 and sector_count.get(sector, 0) < max_per_sector:
        picked_a.append(ticker)
        sector_count[sector] = sector_count.get(sector, 0) + 1

expected = {ticker: stats_a.loc[ticker, 'Expected_Return'] for ticker in picked_a}

weights_a = {}
for ticker in picked_a:
    weights_a[ticker] = expected[ticker] / sum(expected.values())

for i in range(1000):
    extra = 0

    for ticker in picked_a:
        if weights_a[ticker] > max_stock_weight:
            extra += weights_a[ticker] - max_stock_weight
            weights_a[ticker] = max_stock_weight

    for sector in set(sectors[ticker] for ticker in picked_a):
        in_sector = [ticker for ticker in picked_a if sectors[ticker] == sector]
        sector_total = sum(weights_a[ticker] for ticker in in_sector)
        if sector_total > max_sector_weight:
            for ticker in in_sector:
                new_weight = weights_a[ticker] * max_sector_weight / sector_total
                extra += weights_a[ticker] - new_weight
                weights_a[ticker] = new_weight

    if extra == 0:
        break

    has_room = []
    for ticker in picked_a:
        sector_total = sum(weights_a[t] for t in picked_a if sectors[t] == sectors[ticker])
        if weights_a[ticker] < max_stock_weight and sector_total < max_sector_weight:
            has_room.append(ticker)

    room_total = sum(expected[ticker] for ticker in has_room)
    for ticker in has_room:
        weights_a[ticker] += extra * expected[ticker] / room_total


shares_a = {}

money_for_stocks = budget_a

for i in range(100):

    fees_a = 0
    
    for ticker in picked_a:
        shares_a[ticker] = weights_a[ticker] * money_for_stocks / buy_price_cad[ticker]
        fees_a += min(max_fee, fee_per_share * shares_a[ticker]) * exchange_rate

    new_money_for_stocks = budget_a - fees_a

    if abs(new_money_for_stocks - money_for_stocks) < 1e-9:
        break

    money_for_stocks = new_money_for_stocks


rows = []

for ticker in picked_a:
    price = buy_price_cad[ticker]
    if currencies[ticker] == 'USD':
        price = buy_price_cad[ticker] / exchange_rate
    rows.append({'Ticker': ticker, 'Price': price, 'Currency': currencies[ticker],
                 'Shares': shares_a[ticker], 'Value': shares_a[ticker] * buy_price_cad[ticker]})

Portfolio_Final_A = pd.DataFrame(rows)
Portfolio_Final_A['Weight'] = Portfolio_Final_A['Value'] / Portfolio_Final_A['Value'].sum()
Portfolio_Final_A = Portfolio_Final_A.sort_values('Value', ascending=False)
Portfolio_Final_A.index = range(1, len(Portfolio_Final_A) + 1)

purchases_a = Portfolio_Final_A['Value'].sum()

print(Portfolio_Final_A)


     Ticker        Price Currency        Shares          Value    Weight
1     BB.TO    11.100000      CAD  13513.410448  149998.861129  0.150000
2     TD.TO   162.050003      CAD    890.309335  144274.630489  0.144276
3       LLY  1347.109985      USD     63.048623  121824.266452  0.121825
4         C   110.620003      USD    648.141434  102839.177020  0.102840
5      BIIB   249.589996      USD    271.182393   97083.137961  0.097084
6       CAT   510.450012      USD    126.992484   92979.322626  0.092980
7       TXN   263.750000      USD    241.440084   91339.153773  0.091340
8     RY.TO   263.049988      CAD    334.180510   87906.179061  0.087907
9      QCOM   149.940002      USD    293.771357   63180.361284  0.063181
10  SHOP.TO   177.089996      CAD    274.252181   48567.317733  0.048568


### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

In [ ]:
from scipy.optimize import minimize

budget_b = 750_000
number_of_stocks_b = 15
max_stock_weight_b = 0.15
max_sector_weight_b = 0.40

max_fee = 2.15
fee_per_share = 0.001

#Calculate daily returns and volatility for each stock in the filtered list
daily_prices = get_close_prices(
    filtered_tickers, 
    start_date='2024-11-27', 
    end_date='2026-11-27')
daily_returns = daily_prices.pct_change().dropna()
volatility = daily_returns.std()
#Sorting volatility 
volatility = volatility.sort_values(ascending=True)

picked_b = []
sector_count_b = {}
max_stocks_per_sector_b = 6

#look at stocks starting with lowest volatility and add them to portfolio until the number of stocks is reached 
for ticker in volatility.index: 
    if len (picked_b) == number_of_stocks_b:
        break
    sector = sectors[ticker]
    if sector_count_b.get(sector, 0) < max_stocks_per_sector_b:
        picked_b.append(ticker)
        sector_count_b[sector] = (sector_count_b.get(sector, 0) +  1)

#Calculate the covariance matrix for the selected stocks in picked_b
returns_b = daily_returns[picked_b]
covariance_maxtrix = returns_b.cov()
#Optimize weights to minimize portfolio variance while adhering to the constraints
n = len(picked_b)

#Starting with equal weights for each stock in the portfolio
initial_weights = np.array([1/n] * n)
#total weights must sum up to 1 or 100%
constraints_sum = {'type': 'eq', 'fun': lambda weights: np.sum(weights) - 1}

#making sure no sector can exceed 40% of the total portfolio weight
sector_constraints = []
for sector in set(sectors[t] for t in picked_b):
    sector_indices = [i for i in range (n) 
                      if sectors[picked_b[i]] == sector]
    sector_constraints.append({'type': 'ineq', 
                               'fun': lambda weights, indices=sector_indices:
                               max_sector_weight_b - sum(weights[i] for i in indices)})
#calculates variance of the entire portfolio based on weights of each stock and 
#how the stocks move relative to each other
def portfolio_variance(weights, covariance_maxtrix):
    return weights.T @ covariance_maxtrix.values @ weights

#stock must have weight between 0% minimum and 15% maximum of the total portfolio weight
bounds = [(0, max_stock_weight_b)] * n
results = minimize(portfolio_variance, 
                   initial_weights,
                   args=(covariance_maxtrix,),
                   method='SLSQP', 
                   bounds=bounds, 
                   constraints=[constraints_sum] + sector_constraints)

print(results.success)
print(results.message)
print(results.x)

#Match weights to stocks
weights_b = {}
for i, ticker in enumerate(picked_b):
    weights_b[ticker] = results.x[i]

#Calculate shares and transaction fees
shares_b = {}
money_for_stocks_b = budget_b
for iteration in range(100):
    fees_b = 0
    #Number of shares that can be purchased for each stock
    for ticker in picked_b:
        shares_b[ticker] = (weights_b[ticker] * money_for_stocks_b / buy_price_cad[ticker])
        fees_b += min(max_fee, fee_per_share * shares_b[ticker]) * exchange_rate

    new_money_for_stocks_b = budget_b - fees_b

    if abs(new_money_for_stocks_b - money_for_stocks_b) < 0.000001:
        break

    money_for_stocks_b = new_money_for_stocks_b

#Create final portfolio table
rows_b = []
for ticker in picked_b: 
    price_cad = buy_price_cad[ticker]
    if currencies[ticker] == 'USD':
        price = price_cad/exchange_rate
    else: 
        price = price_cad

    value = shares_b[ticker] * price_cad
    rows_b.append({'Ticker': ticker, 
                   'Price': price, 
                   'Currency': currencies[ticker],
                   'Shares': shares_b[ticker], 
                   'Value': value})

Portfolio_Final_B = pd.DataFrame(rows_b)
Portfolio_Final_B['Weight'] = Portfolio_Final_B['Value']/Portfolio_Final_B['Value'].sum()
Portfolio_Final_B = Portfolio_Final_B.sort_values('Value', ascending=False)
Portfolio_Final_B.index = range(1, len(Portfolio_Final_B) + 1)

#Printing of final portfolio 
total_investment_b = Portfolio_Final_B['Value'].sum()
total_fees_b = sum(min(max_fee, fee_per_share * shares_b[ticker]) 
                   * exchange_rate for ticker in picked_b)
total_spent_b = total_investment_b + total_fees_b

print("Client B PORTFOLIO")
print(Portfolio_Final_B)
print("\n Total Stock Investment:", total_investment_b)
print("Total Fees:", total_fees_b)
print("Total Spent:", total_spent_b)
print("Remaining Cash:", budget_b - total_spent_b)

#Check sector weights
sector_weights_b = (Portfolio_Final_B.groupby(
    Portfolio_Final_B['Ticker'].map(sectors)
    )['Weight'].sum())

print("\nSector Weights:")
print(sector_weights_b)

#Portfolio Volatility
final_weights = (Portfolio_Final_B.set_index('Ticker')['Weight'])
portfolio_daily_returns = returns_b @ final_weights
portfolio_volatility = (portfolio_daily_returns.std())
portfolio_volatility_annual = (portfolio_volatility * 252 ** 0.5)
print ("\n Daily Portfolio Volatility:", portfolio_volatility)
print ("Annual Portfolio Volatility:", portfolio_volatility_annual)

print("Daily prices:", daily_prices.shape)
print("Daily returns:", daily_returns.shape)
print("Selected returns:", returns_b.shape)

print("Missing prices:", daily_prices.isna().sum().sum())
print("Missing returns:", returns_b.isna().sum().sum())

ptest_weights = initial_weights.copy()

test_weights[0] += 0.01
test_weights[1] -= 0.01

print("Initial variance:",
      portfolio_variance(initial_weights, covariance_maxtrix))

print("Test variance:",
      portfolio_variance(test_weights, covariance_maxtrix))

print("Optimized variance:", results.fun)
print("Iterations:", results.nit)

True
Optimization terminated successfully
[0.06666667 0.06666667 0.06666667 0.06666667 0.06666667 0.06666667
 0.06666667 0.06666667 0.06666667 0.06666667 0.06666667 0.06666667
 0.06666667 0.06666667 0.06666667]
Client B PORTFOLIO
   Ticker       Price Currency       Shares         Value    Weight
1   RY.TO  263.049988      CAD   190.075062  49999.242822  0.066667
2   TD.TO  162.050003      CAD   308.542067  49999.242822  0.066667
3      KO   96.989998      USD   359.402700  49999.242822  0.066667
4      CL   98.709999      USD   353.140183  49999.242822  0.066667
5     PEP  140.539993      USD   248.032367  49999.242822  0.066667
6     UNP  287.290009      USD   121.335466  49999.242822  0.066667
7      MO   74.870003      USD   465.586562  49999.242822  0.066667
8     AIG   79.680000      USD   437.480761  49999.242822  0.066667
9     BAC   47.040001      USD   741.038828  49999.242822  0.066667
10    USB   56.049999      USD   621.917353  49999.242822  0.066667
11    ABT  103.000000 

NameError: name 'covariance_maxtrtrix' is not defined

### Client B — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

In [ ]:


#For Client B, we will compare the optimized portfolio to an equal-weighted portfolio
#of the same stocks using historical returns and volatility. 
equal_weights = np.array([1/n] * n)
equal_daily_returns = returns_b @ equal_weights

#Historical Return over the period
chosen_return = (1 + portfolio_daily_returns).prod() - 1
equal_return = (1 + equal_daily_returns).prod() - 1

#Annualized Volatility 
chosen_volatility = portfolio_daily_returns.std() * 252 ** 0.5
equal_volatility = equal_daily_returns.std() * 252 ** 0.5

#Comparison table 
comparison_b = pd.DataFrame ({
    'Portfolio': ['Optimized Portfolio', 'Equal Weight Portfolio'],
    'Historical Return': [chosen_return, equal_return],
    'Annualized Volatility': [chosen_volatility, equal_volatility]
})

print(comparison_b)


                Portfolio  Historical Return  Annualized Volatility
0     Optimized Portfolio           0.230227               0.124695
1  Equal Weight Portfolio           0.230227               0.124695


## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

In [ ]:


#For Client B, we will now check the constraints of the optimized portfolio are met. 
print ("Number of stocks:", len(picked_b))
print ("Maximum stock weight:", Portfolio_Final_B['Weight'].max() <= 0.15)
print ("Maximum industry weight:", sector_weights_b.max() <= 0.40)
print ("Weights sum to 100%:,", abs(Portfolio_Final_B['Weight'].sum() - 1) < 0.0001)
print ("Budget passed:", total_spent_b <= budget_b)

Number of stocks: 15
Weights sum to 100%:, True
Maximum stock weight: True
Maximum industry weight: True
Budget passed: False


## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.

In [ ]:
LYN KHINE, 